Squad 2 | Streaming em Tempo Real

**Tabela** | ecommerce_itens_pedido |

**Destino** | squad2.ecommerce_itens_pedido |

**Schema** | id_item_pedido, id_pedido, sku, quantidade, preco_unitario, desconto_aplicado |

**Chave PK** | id_item_pedido |

**Chave FK** | id_pedido → ecommerce_pedidos, sku → ecommerce_produtos |

**Nulos** | Nenhum |

**Depende de** | feat_squad2_99_helpers |

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
import logging
logging.getLogger("azure").setLevel(logging.WARNING)

TABELA        = "ecommerce_itens_pedido"
MODO_GRAVACAO = "overwrite"

inicio = log_inicio(f"feat_squad2_03_ingestao_{TABELA}")


In [0]:
try:
    snapshot_id = get_snapshot_mais_recente()
    log.info(f"Snapshot selecionado: {snapshot_id}")

    df = ler_parquet(snapshot_id, TABELA)
    log.info(f"Leitura OK → {df.count()} linhas | {len(df.columns)} colunas")

except Exception as e:
    log.error(f"Erro ao ler {TABELA}: {str(e)}")
    raise

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

# Schema
log.info("Schema:")
df.printSchema()

# Amostra
display(df)

# Nulos
df_nulos = df.select([
    spark_sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
])

log.info("Nulos por coluna:")
display(df_nulos)

# Validações de negócio
total             = df.count()
total_com_desconto = df.filter(col("desconto_aplicado") > 0).count()
total_sem_desconto = df.filter(col("desconto_aplicado") == 0).count()

log.info(f"Total registros     : {total}")
log.info(f"Com desconto        : {total_com_desconto}")
log.info(f"Sem desconto        : {total_sem_desconto}")

In [0]:
try:
    sucesso = gravar_sql(df, TABELA, mode=MODO_GRAVACAO)

    if sucesso:
        log.info(f"Gravação OK → {get_destino_sql(TABELA)}")
    else:
        raise Exception("Falha na gravação")

except Exception as e:
    log.error(f"Erro ao gravar {TABELA}: {str(e)}")
    raise

In [0]:
try:
    df_sql    = ler_sql(TABELA)
    total_sql = df_sql.count()

    log.info(f"Validação OK → {get_destino_sql(TABELA)}")
    log.info(f"Registros gravados: {total_sql}")

    if total_sql == total:
        log.info(" Origem e destino com mesmo número de registros!")
    else:
        log.warning(
            f" Divergência: "
            f"origem={total} | destino={total_sql}"
        )

    display(df_sql)

except Exception as e:
    log.error(f"Erro na validação: {str(e)}")
    raise

In [0]:
log_fim(f"feat_squad2_03_ingestao_{TABELA}", inicio)